In [ ]:
import pandas as pd
from pathlib import Path

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
file_path = Path("../data/munchee_optimization_dataset.xlsx")

shops_df = pd.read_excel(file_path, sheet_name="Shops")
products_df = pd.read_excel(file_path, sheet_name="Products")
vehicles_df = pd.read_excel(file_path, sheet_name="Vehicles")
routes_df = pd.read_excel(file_path, sheet_name="Routes")
distance_df = pd.read_excel(file_path, sheet_name="Distance_Matrix")

for df in [shops_df, products_df, vehicles_df, routes_df, distance_df]:
    df.columns = df.columns.str.strip().str.lower()

alloc_df = pd.read_csv("../results/tables/ilp_allocation_results.csv")

In [3]:
weights = dict(zip(products_df["product_id"], products_df["carton_weight_kg"]))
volumes = dict(zip(products_df["product_id"], products_df["carton_volume_m3"]))

In [4]:
weights = dict(zip(products_df["product_id"], products_df["carton_weight_kg"]))
volumes = dict(zip(products_df["product_id"], products_df["carton_volume_m3"]))

In [5]:
alloc_df["allocated_weight_kg"] = alloc_df.apply(lambda r: r["allocated_cartons"] * weights[r["product_id"]], axis=1)
alloc_df["allocated_volume_m3"] = alloc_df.apply(lambda r: r["allocated_cartons"] * volumes[r["product_id"]], axis=1)

In [6]:
shop_load_df = alloc_df.groupby("shop_id").agg(
    allocated_cartons=("allocated_cartons", "sum"),
    allocated_weight_kg=("allocated_weight_kg", "sum"),
    allocated_volume_m3=("allocated_volume_m3", "sum")
).reset_index()

In [7]:
shop_load_df = shop_load_df.merge(
    shops_df[["shop_id", "route_id", "shop_name", "town", "service_time_min"]],
    on="shop_id",
    how="left"
)

shop_load_df.head()

,shop_id,allocated_cartons,allocated_weight_kg,allocated_volume_m3,route_id,shop_name,town,service_time_min
0,S001,63,137.6,2.244,R1,Negombo Distributor Outlet,Negombo,18
1,S002,27,59.7,0.968,R1,Seeduwa Stores,Seeduwa,14
2,S003,34,74.7,1.215,R1,Katunayake C&C,Katunayake,14
3,S004,82,177.2,2.885,R1,Ja-Ela Mini Super,Ja-Ela,18
4,S005,39,85.1,1.388,R1,Kandana Mart,Kandana,14


In [ ]:
route_load_df = shop_load_df.groupby("route_id").agg(
    total_cartons=("allocated_cartons", "sum"),
    total_weight_kg=("allocated_weight_kg", "sum"),
    total_volume_m3=("allocated_volume_m3", "sum"),
    total_service_time_min=("service_time_min", "sum"),
    shop_count=("shop_id", "count")
).reset_index()

route_load_df = route_load_df.merge(
    routes_df[["route_id", "route_name", "baseline_route_distance_km", "baseline_route_travel_time_min", "baseline_total_time_with_service_min"]],
    on="route_id",
    how="left"
)

route_load_df

,route_id,total_cartons,total_weight_kg,total_volume_m3,total_service_time_min,shop_count,route_name,baseline_route_distance_km,baseline_route_travel_time_min,baseline_total_time_with_service_min
0,R1,378,825.6,13.449,144,10,Negombo - Colombo North Urban Belt,95.8,253,397
1,R2,303,659.1,10.734,144,10,Gampaha Interior Trade Route,141.2,317,461
2,R3,235,511.1,8.333,144,10,Kurunegala Distribution Corridor,357.5,648,792
3,R4,227,494.3,8.070,144,10,Kegalle - Kandy Highland Route,248.9,548,692
4,R5,297,644.2,10.496,144,10,Western - Southern Coastal Route,334.2,580,724


In [9]:
vehicle_capacity_df = vehicles_df[["vehicle_id", "max_cartons", "max_payload_kg", "max_volume_m3"]].copy()
vehicle_capacity_df

,vehicle_id,max_cartons,max_payload_kg,max_volume_m3
0,L1,760,3300,35
1,L2,680,2950,30


In [10]:
vehicle_state = {
    row["vehicle_id"]: {
        "routes": [],
        "cartons": 0,
        "weight": 0,
        "volume": 0
    }
    for _, row in vehicles_df.iterrows()
}

sorted_routes = route_load_df.sort_values("total_weight_kg", ascending=False)

for _, row in sorted_routes.iterrows():
    best_vehicle = min(vehicle_state.keys(), key=lambda v: vehicle_state[v]["weight"])
    vehicle_state[best_vehicle]["routes"].append(row["route_id"])
    vehicle_state[best_vehicle]["cartons"] += row["total_cartons"]
    vehicle_state[best_vehicle]["weight"] += row["total_weight_kg"]
    vehicle_state[best_vehicle]["volume"] += row["total_volume_m3"]

vehicle_state

{'L1': {'routes': ['R1', 'R3'],
  'cartons': 613,
  'weight': 1336.7,
  'volume': 21.782},
 'L2': {'routes': ['R2', 'R5', 'R4'],
  'cartons': 827,
  'weight': 1797.6000000000001,
  'volume': 29.3}}

In [11]:
assignment_rows = []

for vehicle_id, info in vehicle_state.items():
    assignment_rows.append({
        "vehicle_id": vehicle_id,
        "assigned_routes": ",".join(info["routes"]),
        "total_cartons": info["cartons"],
        "total_weight_kg": info["weight"],
        "total_volume_m3": info["volume"]
    })

vehicle_assignment_df = pd.DataFrame(assignment_rows)
vehicle_assignment_df

,vehicle_id,assigned_routes,total_cartons,total_weight_kg,total_volume_m3
0,L1,"R1,R3",613,1336.7,21.782
1,L2,"R2,R5,R4",827,1797.6,29.300


In [12]:
route_load_df.to_csv("../results/tables/route_load_summary.csv", index=False)
vehicle_assignment_df.to_csv("../results/tables/vehicle_assignment_summary.csv", index=False)